# 07 — Stage 2: Onset Models

Logistic regression models of civil war onset augmented by intervention expectations.

**Inputs**:
- `data/interim/cy_imputed_{1..5}.parquet` — CY baseline covariates (from nb01)
- `data/interim/cy_shadow_{cy}_{ud}.parquet` — shadow measure (from nb06)

**Outputs**:
- `results/tables/tab-topFit.tex`   — in/out-of-sample fit comparison
- `results/tables/tab-logit.tex`    — logit coefficients
- `results/tables/tab-import.tex`   — variable importance
- `results/figures/annualplot.pdf`  — predictions ratio over time
- `results/figures/powersplot.pdf`  — selected-power probabilities

**Reference R scripts**: `17-topNodeFit.R` through `23-introducePredictors.R`

**Models**:
| Name       | Extended variables |
|------------|--------------------|
| Baseline   | F&L (2003) only    |
| Entrants   | E_gov + E_opp      |
| Powers     | + E_major_gov + E_major_opp |
| Neighbors  | + E_contig_gov + E_contig_opp |
| Coethnics  | + E_coethnic_gov + E_coethnic_opp |
| Rulers     | + E_colonial_gov + E_colonial_opp |
| Rivals     | + E_rivals_gov + E_rivals_opp |
| Full       | all of the above |

**Evaluation**: in-sample log-loss + AUC + Vuong test;
out-of-sample leave-one-onset-out log-loss + AUC.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import norm
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import LeaveOneGroupOut

ROOT    = Path("..").resolve()
INTERIM = ROOT / "data" / "interim"
RESULTS = ROOT / "results"
(RESULTS / "tables").mkdir(parents=True, exist_ok=True)
(RESULTS / "figures").mkdir(parents=True, exist_ok=True)

## § 1  Load and merge data

In [ ]:
# Baseline covariates (Fearon & Laitin 2003, Table 1 Model 1)
# Uses lagged lgdp and lpop to match original specification
BASELINE_VARS = [
    "polity2", "lgdp_lag", "lpop_lag",
    "lmtnest", "ncontig", "oil",
    "nwstate", "instab", "prior_war",
    "ethfrac", "relfrac",
    "year",   # controls for number-of-states trend
]

def load_analysis_data() -> pd.DataFrame:
    """Load and average shadow variables across 25 imputations."""
    # Average CY baseline covariates across 5 CY imputations
    cy_frames = [
        pd.read_parquet(INTERIM / f"cy_imputed_{i}.parquet")
        for i in range(1, 6)
    ]
    cy_avg = (
        pd.concat(cy_frames)
        .groupby(["ccode", "year"])[
            ["onset"] + BASELINE_VARS
        ]
        .mean()
        .reset_index()
    )

    # Average shadow variables across 25 imputations
    shadow_cols = [
        "E_gov_asinh", "E_opp_asinh",
        "E_gov_trim_asinh", "E_opp_trim_asinh",
    ]
    sh_frames = [
        pd.read_parquet(INTERIM / f"cy_shadow_{cy}_{ud}.parquet")
        for cy in range(1, 6)
        for ud in range(1, 6)
    ]
    shadow_avg = (
        pd.concat(sh_frames)
        .groupby(["ccode", "year"])[shadow_cols]
        .mean()
        .reset_index()
    )

    return cy_avg.merge(shadow_avg, on=["ccode", "year"], how="left")


data = load_analysis_data()
print(f"Analysis dataset: {len(data):,} country-years, "
      f"{data['onset'].sum():.0f} onsets")

# Identify onset events for leave-one-onset-out CV
data["onset_id"] = (
    data.sort_values(["ccode", "year"])
    .groupby("ccode")["onset"]
    .transform(lambda s: (s.cumsum() * s).replace(0, np.nan))
)
# Non-onset rows get a unique group per country
data["cv_group"] = data["onset_id"].fillna(
    data.groupby("ccode").ngroup() + data["onset_id"].max() + 1
)

## § 2  Model specifications

In [ ]:
def make_specs(data: pd.DataFrame) -> dict[str, list[str]]:
    """Return model specifications as lists of covariate names."""
    base = BASELINE_VARS
    # Core entrants
    entrants = ["E_gov_trim_asinh", "E_opp_trim_asinh"]

    # Heterogeneous utility models require additional aggregation
    # (done in nb06 extended version — TODO: add these aggregates)
    # For now: Entrants only; others are stubs
    return {
        "Baseline": base,
        "Entrants": base + entrants,
        # TODO: Powers, Neighbors, Coethnics, Rulers, Rivals, Full
        # require per-intervener-type aggregation from nb06
    }


SPECS = make_specs(data)

## § 3  Fit models and evaluate

In [ ]:
def prl(y: np.ndarray, y_hat: np.ndarray) -> float:
    """Proportional reduction in log-loss vs. null."""
    null_p  = np.clip(y.mean(), 1e-9, 1 - 1e-9)
    null_ll = -(y * np.log(null_p) + (1 - y) * np.log(1 - null_p)).mean()
    model_ll = log_loss(y, y_hat)
    return (null_ll - model_ll) / null_ll


def fit_and_eval(spec_name: str, covars: list[str]) -> dict:
    sub = data[(["onset", "cv_group"] + covars)].dropna().copy()
    y   = sub["onset"].values
    X   = sm.add_constant(sub[covars].values)

    # In-sample fit
    logit = sm.Logit(y, X).fit(disp=False, maxiter=1000)
    y_hat = logit.predict(X)
    is_ll  = log_loss(y, y_hat)
    is_auc = roc_auc_score(y, y_hat)
    is_prl = prl(y, y_hat)

    # Out-of-fold: leave-one-onset-group-out
    groups = sub["cv_group"].values
    logo   = LeaveOneGroupOut()
    oof_preds = np.zeros(len(y))
    for tr_idx, val_idx in logo.split(X, y, groups):
        try:
            m_fold = sm.Logit(y[tr_idx], X[tr_idx]).fit(disp=False, maxiter=500)
            oof_preds[val_idx] = m_fold.predict(X[val_idx])
        except Exception:
            oof_preds[val_idx] = y[tr_idx].mean()
    oos_ll  = log_loss(y, np.clip(oof_preds, 1e-9, 1 - 1e-9))
    oos_auc = roc_auc_score(y, oof_preds)
    oos_prl = prl(y, np.clip(oof_preds, 1e-9, 1 - 1e-9))

    return {
        "model":   spec_name,
        "n":       len(y),
        "n_onset": int(y.sum()),
        "is_ll":   round(is_ll, 4),
        "is_auc":  round(is_auc, 4),
        "is_prl":  round(is_prl, 4),
        "oos_ll":  round(oos_ll, 4),
        "oos_auc": round(oos_auc, 4),
        "oos_prl": round(oos_prl, 4),
        "logit":   logit,
    }


results = {name: fit_and_eval(name, covars) for name, covars in SPECS.items()}

fit_table = pd.DataFrame([
    {k: v for k, v in r.items() if k != "logit"}
    for r in results.values()
])
print(fit_table.to_string(index=False))

## § 4  Vuong test (Baseline vs. Entrants)

In [ ]:
def vuong_test(model_A_name: str, model_B_name: str) -> tuple[float, float]:
    """Non-nested Vuong test comparing two logit models on their common sample.

    Refits both models on the intersection of their non-missing rows so that
    predictions are on the same observations.  Returns (z, p): z > 0 favours A.
    """
    covars_A = SPECS[model_A_name]
    covars_B = SPECS[model_B_name]

    mask_A = data[["onset"] + covars_A].notna().all(axis=1)
    mask_B = data[["onset"] + covars_B].notna().all(axis=1)
    common = data.index[mask_A & mask_B]

    sub = data.loc[common].copy()
    y   = sub["onset"].values

    X_A = sm.add_constant(sub[covars_A].values)
    X_B = sm.add_constant(sub[covars_B].values)

    m_A = sm.Logit(y, X_A).fit(disp=False, maxiter=1000)
    m_B = sm.Logit(y, X_B).fit(disp=False, maxiter=1000)

    p_A = np.clip(m_A.predict(X_A), 1e-9, 1 - 1e-9)
    p_B = np.clip(m_B.predict(X_B), 1e-9, 1 - 1e-9)

    ll_A = y * np.log(p_A) + (1 - y) * np.log(1 - p_A)
    ll_B = y * np.log(p_B) + (1 - y) * np.log(1 - p_B)
    m    = ll_A - ll_B
    n    = len(y)
    z    = np.sqrt(n) * m.mean() / m.std()
    p    = 2 * (1 - norm.cdf(abs(z)))
    return z, p


if "Entrants" in SPECS and "Baseline" in SPECS:
    z, p = vuong_test("Entrants", "Baseline")
    print(f"Vuong test (Entrants vs Baseline): z = {z:.3f}, p = {p:.4f}")
